# 1. Tensors and data flow
### 1.1 Tensor

#### What it is  
A tensor is the core data structure in deep learning: a multi-dimensional array with a defined shape and data type. In PyTorch, everything—inputs, weights, activations, gradients—is a tensor.

For your segmentation model, the main tensor shapes are:

- Input images: (B,C,H,W)

    - B: batch size (number of patches in one step)

    - C: channels (e.g. 8: DEM + derivatives)

    - H,W: spatial dimensions of the patch

- Masks: (B,H,W)

    - Each pixel holds an integer class index [0,num_classes−1]

- Model outputs (logits): (B,num_classes,H,W)

#### Why it matters  
Understanding tensor shapes is the foundation. Every layer you use—convolutions, pooling, upsampling—transforms these shapes in specific ways. If you know the shapes, you can reason about the architecture.

 ---
### 1.2 Batch

What it is  
A batch is a group of samples processed together in one forward/backward pass.

Why it matters

- It stabilizes gradient estimates (compared to single samples).

- It allows efficient use of GPU parallelism.

- Many operations (BatchNorm, loss) are defined over the batch dimension.

 ---
 ---
 ---
# 2. Convolution and feature extraction

Now we follow the first thing that happens to your input tensor inside the model: convolutions.

### 2.1 Convolution (Conv2D)

#### What it is  
A 2D convolution applies learnable filters (kernels) over the spatial dimensions of the input. Each filter “looks” at a local neighborhood and produces one output channel.

Formally, for one output channel:
y(i,j)=∑c=1Cin∑u,vWc,u,v⋅xc(i+u,j+v)+b

Where:

- x is the input tensor

- W is the kernel (weights)

- b is the bias

- Cin is the number of input channels

Intuition

- Early layers learn edge/texture detectors.

- Deeper layers learn more abstract patterns (shapes, regions, structures).

Variants

- Standard Conv2D 

- Dilated convolutions (larger receptive field without pooling)

- Depthwise separable convolutions (more efficient)

U-Net ConvBlock is essentially:

- Conv2D → BatchNorm → Activation → Conv2D → BatchNorm → Activation

This is the basic feature extraction unit used in both encoder and decoder.

 ---
### 2.2 Padding

#### What it is  
Padding adds extra pixels around the border of the input (often zeros) so that the spatial size is preserved after convolution.

Without padding, a 3×3 convolution reduces H,W by 2.

Why it matters in UNet

- You want encoder and decoder feature maps to align spatially for skip connections.

- Using padding=1 with kernel_size=3 keeps H,W unchanged.

 ---
### 2.3 Stride

#### What it is  
Stride is how far the convolution kernel moves each step.

- Stride = 1 → dense scanning

- Stride > 1 → downsampling

You mostly use stride = 1 in convolutions, and use pooling for downsampling instead.

 ---
 ---
 ---
# 3. Activation functions

After each convolution, you apply an activation function. This is where non-linearity enters the model.

### 3.1 Why activation functions exist

Without activations, a stack of linear layers (convolutions) is still just a linear function. It cannot model complex, non-linear relationships.

Activation functions allow the network to approximate arbitrary non-linear mappings.

 ---
### 3.2 ReLU

#### Definition
ReLU(x)=max⁡(0,x)

Properties

- Simple and fast

- Encourages sparsity (many zeros)

- But can cause “dead neurons” if weights push activations permanently negative.

 ---
### 3.3 LeakyReLU

#### Definition
LeakyReLU(x)={xx>0αxx≤0}

with α typically small (e.g. 0.01 or 0.1).

Why it might be preferable

- It keeps a small gradient for negative values.

- Reduces the risk of dead neurons.

- Can be more robust when inputs are noisy or have wide value ranges.

 ---
 ---
 ---
# 4. Normalization and regularization inside the model

Now we look at the “stabilizers” inside your blocks: BatchNorm and Dropout.

### 4.1 Batch Normalization (BatchNorm2d)

#### What it does  
For each channel, over the batch and spatial dimensions, it normalizes activations to have mean 0 and variance 1, then scales and shifts them with learnable parameters.

Formally, for each channel:
x^=x−μσ2+ϵ,y=γx^+β

Why it matters

- Reduces internal covariate shift (distribution drift of activations).

- Allows higher learning rates.

- Often speeds up and stabilizes training.

ConvBlock uses BatchNorm after convolutions. This helps when your input channels have very different distributions.

 ---
### 4.2 Dropout2D

#### What it does  
Randomly zeroes entire feature channels (not just individual pixels) during training.

Why it matters

- Forces the network not to rely too heavily on any single channel.

- Reduces overfitting, especially when positive class (rivers) is extremely rare.

- In 2D feature maps, dropping whole channels is more structured than dropping random pixels.
  
You can use Dropout2d in ConvBlock. This is particularly relevant when dataset is highly imbalanced: most pixels are background for example.

 ---
 ---
 ---
# 5. Encoder: downsampling and hierarchical features

Now we follow the data as it moves deeper into the network.

### 5.1 Max Pooling (MaxPool2d)

#### What it does  
For each channel, it takes the maximum value in non-overlapping windows (e.g. 2×2), reducing spatial resolution by a factor of 2.

#### Why it matters

- Increases the receptive field (each deeper neuron “sees” a larger part of the input).

- Keeps the most salient activations.

- Reduces memory and computation.

 ---
### 5.2 EncoderBlock

The EncoderBlock does:

- ConvBlock(in_channels, out_channels)

- MaxPool2d(kernel_size=2, stride=2)

It returns:

- skip: output of ConvBlock (high-resolution features)

- pooled: output after pooling (lower resolution, deeper features)

#### Why this structure?

- skip is used later in the decoder to recover fine spatial details.

- pooled is passed deeper into the network to build more abstract representations.

#### Big picture  
The encoder gradually transforms the input into a hierarchy of features:

- Shallow layers: local edges, textures, small patterns

- Deeper layers: shapes, regions, global context

 ---
 ---
 ---
# 6. Bottleneck: deepest representation
### 6.1 Bottleneck ConvBlock

At the bottom of the UNet, you have:

- self.bottleneck = ConvBlock(512, 1024)

What it represents

- Lowest spatial resolution

- Highest number of channels

- Most abstract representation of the input

#### Why it matters

- This is where the model can integrate global context.

- For segmentation, this helps understand “where” structures are in relation to the whole patch.

 ---
 ---
 ---
# 7. Decoder: upsampling and reconstruction

Now we go back up: the decoder reconstructs a high-resolution segmentation map.

### 7.1 Upsampling

You have two main options conceptually:

- Bilinear upsampling (nn.Upsample)

    - Non-learnable

    - Smooth interpolation

    - No checkerboard artifacts

- Transposed convolution (nn.ConvTranspose2d)

    - Learnable

    - Can produce sharper boundaries

    - Risk of checkerboard artifacts if not carefully designed

You can create a DecoderBlock supporting both (via bilinear=True/False).

 ---
### 7.2 Spatial alignment and padding

After upsampling, the spatial size of the decoder feature map might not perfectly match the skip connection (due to odd dimensions, pooling, etc.).

You handle this by computing:

- diff_y = skip.size(2) - x.size(2)

- diff_x = skip.size(3) - x.size(3)

And then padding x so it matches skip.

#### Why this matters

- Concatenation requires exact spatial alignment.

- Without this, you’d get shape mismatch errors.

- In geospatial data, odd tile sizes are common, so this is practically important.

 ---
### 7.3 Skip connections and concatenation

Once sizes match, you do:

- x = torch.cat([x, skip], dim=1)

What this does

- Concatenates the upsampled decoder features with the encoder’s high-resolution features along the channel dimension.

#### Why it’s powerful

- Decoder alone would only have coarse, abstract features.

- Skip connections inject fine-grained spatial detail from the encoder.

- This combination allows UNet to produce sharp segmentations even after heavy downsampling.

 ---
### 7.4 DecoderBlock

So a DecoderBlock is:

- Upsample decoder features

- Align with skip (padding)

- Concatenate with skip

- Apply ConvBlock to fuse and refine

This pattern is repeated until you reach the original resolution.

 ---
 ---
 ---
# 8. Output layer and logits
### 8.1 1×1 Convolution

The final layer is:

self.output = nn.Conv2d(64, num_classes, kernel_size=1)

#### What it does

- For each pixel, it takes the 64-dimensional feature vector and linearly maps it to num_classes logits.

#### Why 1×1?

- It doesn’t mix spatial information, only channels.

- It acts as a per-pixel classifier on top of the learned features.

 ---
### 8.2 Logits and softmax

The model returns raw logits: (B,num_classes,H,W).

- During training, these logits are passed directly to the loss function (e.g. CrossEntropy, Focal).

- The loss function internally applies log_softmax or similar.

- During inference, you typically apply softmax and then argmax to get class predictions.

 ---
 ---
 ---
# 9. Loss functions and class imbalance

Now we move outside the model into the training loop.

### 9.1 CrossEntropyLoss

What it assumes

- Classes are reasonably balanced.

- Every pixel is equally important.

Why it fails for thin objects

- Background dominates.

- The model can get high accuracy by predicting background everywhere.

- It doesn’t care about the shape or continuity of small structures.

 ---
### 9.2 Dice loss

#### What it measures

Dice coefficient measures overlap between prediction and ground truth:
Dice=2∣P∩G∣∣P∣+∣G∣

Dice loss is 1−Dice.

Why it’s good for thin objects

- It directly optimizes overlap, not per-pixel accuracy.

- It is less sensitive to class imbalance.

 ---
### 9.3 Focal loss

#### What it does

Focal loss modifies CrossEntropy to focus more on hard examples:

- FL=(1−pt)γ⋅CE

Where pt is the predicted probability of the true class.

#### Why it helps

- Easy examples (background) are down-weighted.

- Hard examples (rivers) contribute more to the loss.

 ---
### 9.4 Combined Dice + Focal

In the loop, you can use:

- loss = 0.5 * focal(pred, target) + 0.5 * dice_loss(pred, target)

#### Why this is powerful

- Dice focuses on overlap and shape.

- Focal focuses on hard pixels and class imbalance.

- Together, they encourage the model to both detect and correctly shape thin structures.

 ---
 ---
 ---
# 10. Training loop mechanics
### 10.1 Forward pass

For each batch:

- Load images and masks

- Move them to device

- Compute outputs = model(images)

- Compute loss = loss_fn(outputs, masks)

This is the forward pass: data flows through the model, loss is computed.

 ---
### 10.2 Backward pass and optimization

Then:

- optimizer.zero_grad() — clear old gradients

- loss.backward() — compute gradients via backpropagation

- optimizer.step() — update parameters using gradients

Backpropagation

- Uses the chain rule to propagate gradients from the loss back through all layers.

- Each parameter gets a gradient telling it how to change to reduce the loss.

 ---
### 10.3 Epochs and validation

An epoch is one full pass over the training dataset.

At the end of each epoch, you:

- Run a validation loop (no gradients)

- Compute validation loss and Dice score

- Optionally save a checkpoint if validation loss improves

This gives you a signal of generalization, not just training performance.

 ---
 ---
 ---
# 11. Dataset and DataLoader
### 11.1 Dataset

The PatchDataset:

- Scans folders for img*.npy and mask*.npy pairs

- Loads them as NumPy arrays

- Converts them to PyTorch tensors

- Returns (image, mask) for each index

This defines what one sample is.

 ---
### 11.2 DataLoader

DataLoader wraps the dataset and handles:

- Batching

- Shuffling

- Parallel loading (num_workers)

- Pinned memory for faster GPU transfer

This defines how samples are fed into the model.

 ---
 ---
 ---
# 12. The big picture: how it all connects

Let’s tie it all together in one flow:

- Dataset loads raw patches → tensors (B,C,H,W) and masks (B,H,W).

- DataLoader batches them and feeds them to the model.

- UNet encoder:

    - ConvBlocks extract local features.

    - BatchNorm stabilizes activations.

    - LeakyReLU introduces non-linearity.

    - Dropout2D regularizes.

    - MaxPool downsamples and builds a feature hierarchy.

- Bottleneck integrates global context at low resolution.

- UNet decoder:

    - Upsampling restores spatial resolution.

    - Skip connections inject high-resolution details.

    - ConvBlocks fuse deep and shallow features.

- Output layer (1×1 conv) maps features to class logits per pixel.

- Loss functions (Dice + Focal) compare logits to ground truth masks, focusing on overlap and hard pixels.

- Backpropagation computes gradients through all layers.

- Optimizer (Adam) updates weights to reduce loss.

- Validation measures performance (loss + Dice) on unseen patches.

- Checkpoints save the best model based on validation loss.